# PAPNet training code



## 1. Setup

Define the desired repository path to be. This is where the `pointcloud-bench` project repository is cloned

In [15]:
REPO_PATH = "/content/pointcloud-bench"

### 1.1 Clone the working repository

In [2]:
!git clone --recursive --branch papnet-occlusion https://github.com/DavidClaszen/pointcloud-bench {REPO_PATH}

fatal: destination path '/content/pointcloud-bench' already exists and is not an empty directory.


`cd` to the repo then pull the latest changes to the repository (if there is)

In [3]:
%cd {REPO_PATH}
!git pull && git submodule update --recursive

/content/pointcloud-bench
Already up to date.


### 1.2 Mount the dataset from Google Drive

In [4]:
import os
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


Inside the dataset folder, we expect the presence of `fullmodelnet40.tar`

In [ ]:
# Define the path to the dataset in Google Drive where the data is saved (fullmodelnet40.tar.gz, partialmodelnet40.tar.gz)
dataset_path = '/content/drive/MyDrive/_Temp/cs7643-final-proj/pointcloud-bench/datasets'
output_path = '/content/drive/MyDrive/_Temp/cs7643-final-proj/pointcloud-bench/checkpoints/papnet'

# Check contents to make sure it's there
file_exists = lambda path: os.path.exists(path)
assert file_exists(os.path.join(dataset_path, 'fullmodelnet40.tar.gz'))
assert file_exists(os.path.join(dataset_path, 'partialmodelnet40.tar.gz'))

### 1.2 Install Requirements

In [6]:
%pip install -r envs/papnet/requirements.txt  --no-deps

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu126
  Cloning https://github.com/AMLab-Amsterdam/lie_learn (to revision master) to /tmp/pip-install-b39wun4l/lie-learn_0a06797c20a643929cd7b78182c58253
  Running command git clone --filter=blob:none --quiet https://github.com/AMLab-Amsterdam/lie_learn /tmp/pip-install-b39wun4l/lie-learn_0a06797c20a643929cd7b78182c58253
  Resolved https://github.com/AMLab-Amsterdam/lie_learn to commit edf012f5f60af320175d2e6269db78b984b5bfc3
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


### 1.3 Copy and extract the data to local working directory

In [ ]:
!rsync -avP {dataset_path}/fullmodelnet40.tar.gz {REPO_PATH}/datasets
!rsync -avP {dataset_path}/partialmodelnet40.tar.gz {REPO_PATH}/datasets
!tar -xvzf {REPO_PATH}/datasets/fullmodelnet40.tar.gz -C {REPO_PATH}/datasets
!tar -xvzf {REPO_PATH}/datasets/partialmodelnet40.tar.gz -C {REPO_PATH}/datasets

sending incremental file list

sent 68 bytes  received 12 bytes  160.00 bytes/sec
total size is 1,833,210,387  speedup is 22,915,129.84
sending incremental file list
partialmodelnet40.tar.gz
  2,162,130,396 100%  275.29MB/s    0:00:07 (xfr#1, to-chk=0/1)

sent 2,162,658,366 bytes  received 35 bytes  254,430,400.12 bytes/sec
total size is 2,162,130,396  speedup is 1.00
fullmodelnet40/
fullmodelnet40/test_filenames.txt
fullmodelnet40/test_gt_rot.npy
fullmodelnet40/test_gt_tra.npy
fullmodelnet40/test_labels.npy
fullmodelnet40/test_points.npy
fullmodelnet40/train_filenames.txt
fullmodelnet40/train_gt_rot.npy
fullmodelnet40/train_gt_tra.npy
fullmodelnet40/train_labels.npy
fullmodelnet40/train_points.npy
partialmodelnet40/
partialmodelnet40/test_labels.npy
partialmodelnet40/test_gt_tra.npy
partialmodelnet40/test_gt_rot.npy
partialmodelnet40/partialmodelnet40_shape_names.txt
partialmodelnet40/partialmodelnet40_test.txt
partialmodelnet40/partialmodelnet40_train.txt
partialmodelnet40/train_poin

## 2. Training


The code saves the runs and the checkpoints to `{dataset_path}`. This saves data to Drive, keeping the data even after the Colab runtime shuts down.

In [10]:
import sys

### 2.1 Training on full dataset

Let us run the `train.py` script to **train** PAPNet on the Fullmodelnet40 dataset.

Save location after training:
- checkpoints per epoch: `{dataset_path}/fullmodelnet40_reg_ckpts`
- checkpoints per run: `{dataset_path}/fullmodelnet40_reg_runs`

In [ ]:
%cd {REPO_PATH}/repos/PAPNet
!python train.py --dataset pm40 --data_path {REPO_PATH}/datasets/fullmodelnet40 --batch_size=128 --ckpt_path={output_path}/fullmodelnet40_reg_ckpts --runs_path={output_path}/fullmodelnet40_reg_runs --weight_decay=0.01

### 2.2 Training on full dataset with occlusions

Let us run the `train.py` script to **train** PAPNet on the Fullmodelnet40 dataset, with occlusions 0.01~0.90

Save location after training:
- checkpoints per epoch: `{dataset_path}/fullmodelnet40_occlusion_reg_ckpts`
- checkpoints per run: `{dataset_path}/fullmodelnet40_occlusion_reg_runs`

In [ ]:
%cd {REPO_PATH}/repos/PAPNet
!python train.py --dataset pm40 --data_path {REPO_PATH}/datasets/fullmodelnet40 --batch_size=128 --ckpt_path={output_path}/fullmodelnet40_occlusion_reg_ckpts --runs_path={output_path}/fullmodelnet40_occlusion_reg_runs --weight_decay=0.01 --occlusion_min=0.01 --occlusion_max=0.90


/content/pointcloud-bench/repos/PAPNet
The size of train data is 98430
The size of test data is 2468
# classifier parameters: 1154874
  0% 0/769 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:5167: UserWarning: Default grid_sample and affine_grid behavior has changed to align_corners=False since 1.3.0. Please specify align_corners=True if the old behavior is desired. See the documentation of grid_sample for details.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/

### 2.3 Training on partial dataset with occlusions

Let us run the `train.py` script to **train** PAPNet on the Partialmodelnet40 dataset, with occlusions 0.01~0.80

Save location after training:
- checkpoints per epoch: `{dataset_path}/partialmodelnet40_occlusion_reg_ckpts`
- checkpoints per run: `{dataset_path}/partialmodelnet40_occlusion_reg_runs`

In [19]:
%ls /content/pointcloud-bench/datasets/partialmodelnet40


partialmodelnet40_shape_names.txt  test_gt_tra.npy   train_gt_tra.npy
partialmodelnet40_test.txt         test_labels.npy   train_labels.npy
partialmodelnet40_train.txt        test_points.npy   train_points.npy
test_gt_rot.npy                    train_gt_rot.npy


In [ ]:
%cd {REPO_PATH}/repos/PAPNet
!python train.py --dataset pm40 --data_path {REPO_PATH}/datasets/partialmodelnet40 --batch_size=128 --ckpt_path={output_path}/partialmodelnet40_occlusion_reg_ckpts --runs_path={output_path}/partialmodelnet40_occlusion_reg_runs --occlusion_min=0.01 --occlusion_max=0.80


/content/pointcloud-bench/repos/PAPNet
The size of train data is 98430
The size of test data is 2468
compute 0.pkl.gz... save 0.pkl.gz... done
compute 0.pkl.gz... save 0.pkl.gz... done
compute 1.pkl.gz... save 1.pkl.gz... done
compute 1.pkl.gz... save 1.pkl.gz... done
compute 2.pkl.gz... save 2.pkl.gz... done
compute 2.pkl.gz... save 2.pkl.gz... done
compute 3.pkl.gz... save 3.pkl.gz... done
compute 3.pkl.gz... save 3.pkl.gz... done
compute 4.pkl.gz... save 4.pkl.gz... done
compute 5.pkl.gz... save 5.pkl.gz... done
compute 6.pkl.gz... save 6.pkl.gz... done
compute 7.pkl.gz... save 7.pkl.gz... done
compute 8.pkl.gz... save 8.pkl.gz... done
compute 9.pkl.gz... save 9.pkl.gz... done
compute 10.pkl.gz... save 10.pkl.gz... done
compute 11.pkl.gz... save 11.pkl.gz... done
compute 12.pkl.gz... save 12.pkl.gz... done
compute 13.pkl.gz... save 13.pkl.gz... done
compute 14.pkl.gz... save 14.pkl.gz... done
compute 4.pkl.gz... save 4.pkl.gz... done
compute 15.pkl.gz... save 15.pkl.gz... done
compu

In [ ]:
# Let us continue training to improve performance
%cd {REPO_PATH}/repos/PAPNet
!python train.py --dataset pm40 --data_path {REPO_PATH}/datasets/partialmodelnet40 --nepoch=30 --batch_size=128 --ckpt_path={output_path}/partialmodelnet40_occlusion_reg_ckpts_cont --runs_path={output_path}/partialmodelnet40_occlusion_reg_runs_cont --occlusion_min=0.01 --occlusion_max=0.80 --model_path={output_path}/partialmodelnet40_occlusion_reg_ckpts/pm40/model_20_0.7925445705024311.pth


/content/pointcloud-bench/repos/PAPNet
The size of train data is 98430
The size of test data is 2468
# classifier parameters: 1154874
  0% 0/769 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:5167: UserWarning: Default grid_sample and affine_grid behavior has changed to align_corners=False since 1.3.0. Please specify align_corners=True if the old behavior is desired. See the documentation of grid_sample for details.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/